<a href="https://colab.research.google.com/github/Innovatewithapple/keras/blob/main/transformerpractice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input,Dense,Embedding,Layer
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [13]:
subjects = ["I", "You", "He", "She", "We", "They"]
verbs = ["eat", "like", "see", "have", "want", "buy", "find", "need"]
objects = ["apple", "car", "book", "coffee", "house", "dog", "cat", "food"]

# English → Spanish mapping (simple)
translation = {
    "I": "Yo", "You": "Tú", "He": "Él", "She": "Ella", "We": "Nosotros", "They": "Ellos",
    "eat": "como", "like": "gusta", "see": "veo", "have": "tengo",
    "want": "quiero", "buy": "compro", "find": "encuentro", "need": "necesito",
    "apple": "manzana", "car": "coche", "book": "libro", "coffee": "café",
    "house": "casa", "dog": "perro", "cat": "gato", "food": "comida"
}

input_texts = []
target_texts = []

for s in subjects:
    for v in verbs:
        for o in objects:
            eng = f"{s} {v} {o}"
            spa = f"{translation[s]} {translation[v]} {translation[o]}"
            input_texts.append(eng)
            target_texts.append(spa)

print(len(input_texts))  # 6 * 8 * 8 = 384

384


In [14]:
extra_phrases = [
    ("Good morning", "Buenos días"),
    ("Thank you", "Gracias"),
    ("I am happy", "Estoy feliz"),
    ("Where is the car", "Dónde está el coche"),
    ("I need help", "Necesito ayuda"),
    ("See you soon", "Hasta pronto"),
    ("I am learning", "Estoy aprendiendo"),
    ("The sky is blue", "El cielo es azul"),
    ("The water is cold", "El agua está fría"),
    ("It is hot today", "Hace calor hoy")
]

for eng, spa in extra_phrases:
    input_texts.append(eng)
    target_texts.append(spa)

In [15]:
input_texts = input_texts
target_texts = target_texts

target_texts = ['startseq ' + t + ' endseq' for t in target_texts]

In [17]:
#Tokenization
input_tokenization = Tokenizer()
input_tokenization.fit_on_texts(input_texts)
input_sequences = input_tokenization.texts_to_sequences(input_texts)

output_tokenization = Tokenizer()
output_tokenization.fit_on_texts(target_texts)
output_sequences = output_tokenization.texts_to_sequences(target_texts)

In [20]:
input_vocab_size = len(input_tokenization.word_index) + 1
output_vocab_size = len(output_tokenization.word_index) + 1

In [19]:
max_input_length = max(len(seq) for seq in input_sequences)
max_output_length = max(len(seq) for seq in output_sequences)

In [21]:
input_sequences = pad_sequences(input_sequences,maxlen=max_input_length,padding='post')
output_sequences = pad_sequences(output_sequences,maxlen=max_output_length,padding='post')

In [23]:
input_sequences.shape

(394, 4)

In [24]:
#this is for training
decoder_input= output_sequences[:,:-1]
decoder_output = output_sequences[:,1:]

decoder_output = np.expand_dims(decoder_output,-1)

In [28]:
a = Input(shape=(max_input_length,))
a

<KerasTensor shape=(None, 4), dtype=float32, sparse=False, ragged=False, name=keras_tensor_1>

In [31]:
class Attention(Layer):
  def __init__(self,d_model):
      super().__init__()
      self.Wq = Dense(d_model)
      self.Wk = Dense(d_model)
      self.Wv = Dense(d_model)

  def call(self,q,k,v):
    Q = self.Wq(q)
    K = self.Wk(k)
    V = self.Wv(v)

    scores = tf.matmul(Q,K,transpose_b=True)
    dk = tf.cast(tf.shape(K)[-1],tf.float32)
    scores = scores / tf.sqrt(dk)

    weights = tf.nn.softmax(scores)
    output = tf.matmul(weights,V)

    return output

In [33]:
# self-attention
encoder_input = Input(shape=(max_input_length,))
x = Embedding(input_vocab_size,64)(encoder_input)

encoder_output = Attention(64)(x,x,x)

decoder_inputs = Input(shape=(max_output_length -1,)) # we do -1 here so it would not pick last word because we have to predict that word
y = Embedding(output_vocab_size,64)(decoder_inputs)

decoder_outputs = Attention(64)(y,y,y)

#cross-attention
final_decoder = Attention(64)(decoder_outputs,encoder_output,encoder_output)

output = Dense(output_vocab_size,activation='softmax')(final_decoder)

model = Model([encoder_input,decoder_inputs],output)

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit([input_sequences,decoder_input],decoder_output,epochs=200,validation_split=0.2)

Epoch 1/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 72ms/step - accuracy: 0.1778 - loss: 3.7478 - val_accuracy: 0.2101 - val_loss: 3.6989
Epoch 2/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.2000 - loss: 3.5027 - val_accuracy: 0.2101 - val_loss: 3.4323
Epoch 3/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.2000 - loss: 3.0644 - val_accuracy: 0.2152 - val_loss: 3.2314
Epoch 4/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.2000 - loss: 2.8788 - val_accuracy: 0.2000 - val_loss: 3.3235
Epoch 5/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.2000 - loss: 2.7850 - val_accuracy: 0.2152 - val_loss: 3.4104
Epoch 6/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.2000 - loss: 2.7348 - val_accuracy: 0.2000 - val_loss: 3.4836
Epoch 7/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.2000 - loss: 2.7018 - val_accuracy: 0.2152 - val_loss: 3.5404
Epoch 8/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.2000 - loss: 2.6693 - val_accuracy: 0.